In [14]:
import pandas as pd
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

In [15]:
# 📄 2. Cargar datasets
df_sellin = pd.read_csv("sell-in.txt", sep="\t")
df_productos = pd.read_csv("tb_productos.txt", sep="\t")
tb_stocks = pd.read_csv("tb_stocks.txt", sep="\t")

In [16]:
tb_stocks

,periodo,product_id,stock_final
0,201810,20524,1.61267
1,201810,20311,2.93657
2,201810,20654,6.83269
3,201810,21005,1.01338
4,201810,20974,0.34595
...,...,...,...
13686,201912,20453,1.43741
13687,201912,21026,7.26817
13688,201912,21054,0.50833
13689,201912,20981,2.18491


In [17]:
df_productos

,cat1,cat2,cat3,brand,sku_size,product_id,descripcion
0,FOODS,ADEREZOS,Aji Picante,NATURA,240,20609,Salsa Aji Picante
1,FOODS,ADEREZOS,Barbacoa,NATURA,250,20266,Salsa Barbacoa
2,FOODS,ADEREZOS,Barbacoa,NATURA,400,20325,Salsa Barbacoa
3,FOODS,ADEREZOS,Barbacoa,NATURA,500,20503,Salsa Barbacoa
4,FOODS,ADEREZOS,Chimichurri,NATURA,350,20797,Chimichurri
...,...,...,...,...,...,...,...
1246,REF,TE,Frutas,TWININGS,20,21271,Frutas
1247,REF,TE,Hierbas,TWININGS,20,21202,Manzanilla
1248,REF,TE,Hierbas,TWININGS,20,21218,Menta
1249,REF,TE,Verde,TWININGS,20,21192,Verde


In [18]:
    # 📄 Leer lista de productos a predecir
with open("product_id_apredecir201912.txt", "r") as f:
    product_ids = [int(line.strip()) for line in f if line.strip().isdigit()]

In [19]:
df_sellin['timestamp'] = pd.to_datetime(df_sellin['periodo'], format='%Y%m')

In [20]:
# Filtrar hasta dic 2019 y productos requeridos
df_filtered = df_sellin[
    (df_sellin['timestamp'] <= '2019-12-01') &
    (df_sellin['product_id'].isin(product_ids))
]
df_filtered

,periodo,customer_id,product_id,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,timestamp
0,201701,10234,20524,0,2,0.05300,0.05300,2017-01-01
1,201701,10032,20524,0,1,0.13628,0.13628,2017-01-01
2,201701,10217,20524,0,1,0.03028,0.03028,2017-01-01
3,201701,10125,20524,0,1,0.02271,0.02271,2017-01-01
4,201701,10012,20524,0,11,1.54452,1.54452,2017-01-01
...,...,...,...,...,...,...,...,...
2945813,201912,10105,20853,0,1,0.02230,0.02230,2019-12-01
2945814,201912,10092,20853,0,1,0.00669,0.00669,2019-12-01
2945815,201912,10006,20853,0,7,0.02898,0.02898,2019-12-01
2945816,201912,10018,20853,0,4,0.01561,0.01561,2019-12-01


In [21]:
df_monthly_product = df_filtered.groupby(['timestamp', 'product_id'], as_index=False).aggregate(
    {
        'tn': 'sum',
        "plan_precios_cuidados": "max",
        "cust_request_qty": "sum",
        "cust_request_tn": "sum",

    }
)
# le mergeo a df_monthly_product los datos de tb_products excepto descripcion usando product_id
df_monthly_product = df_monthly_product.merge(
    df_productos, on='product_id', how='left')
df_monthly_product = df_monthly_product.drop(columns=['descripcion'])
# mergeo tb_stocks usando periodo y product_id
tb_stocks['timestamp'] = pd.to_datetime(tb_stocks['periodo'], format='%Y%m')
df_monthly_product = df_monthly_product.merge(
    tb_stocks[['timestamp', 'product_id', 'stock_final']],
    on=['timestamp', 'product_id'],
    how='left'
)
df_monthly_product

,timestamp,product_id,tn,plan_precios_cuidados,cust_request_qty,cust_request_tn,cat1,cat2,cat3,brand,sku_size,stock_final
0,2017-01-01,20001,934.77222,0,479,937.72717,HC,ROPA LAVADO,Liquido,ARIEL,3000,NaN
1,2017-01-01,20002,550.15707,0,391,555.18654,HC,ROPA LAVADO,Liquido,LIMPIEX,3000,NaN
2,2017-01-01,20003,1063.45835,0,438,1067.81543,FOODS,ADEREZOS,Mayonesa,NATURA,475,NaN
3,2017-01-01,20004,555.91614,0,339,569.37394,FOODS,ADEREZOS,Mayonesa,NATURA,240,NaN
4,2017-01-01,20005,494.27011,0,249,494.60084,FOODS,ADEREZOS,Mayonesa,NATURA,120,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
22344,2019-12-01,21263,0.01270,0,5,0.01270,PC,CABELLO,SHAMPOO,VICHY,250,0.50420
22345,2019-12-01,21265,0.05007,0,5,0.05007,PC,PIEL1,CUIDADO ESPECIAL,LANCOME,32,0.22068
22346,2019-12-01,21266,0.05121,0,6,0.05121,PC,PIEL1,CUIDADO ESPECIAL,LANCOME,32,0.11603
22347,2019-12-01,21267,0.01569,0,4,0.01569,PC,PIEL1,Cara,NIVEA,250,0.54007


In [22]:
# transformo cat1, cat2, cat3, brand y sku_size a categoricos
df_monthly_product['cat1'] = df_monthly_product['cat1'].astype('category')
df_monthly_product['cat2'] = df_monthly_product['cat2'].astype('category')
df_monthly_product['cat3'] = df_monthly_product['cat3'].astype('category')
df_monthly_product['brand'] = df_monthly_product['brand'].astype('category')
df_monthly_product['sku_size'] = df_monthly_product['sku_size'].astype('category')


In [23]:
static_features_df = pd.DataFrame({
    'cat1': df_monthly_product.groupby('product_id')['cat1'].first(),
    'cat2': df_monthly_product.groupby('product_id')['cat2'].first(),
    'cat3': df_monthly_product.groupby('product_id')['cat3'].first(),
    'brand': df_monthly_product.groupby('product_id')['brand'].first(),
    'sku_size': df_monthly_product.groupby('product_id')['sku_size'].first(),
}).reset_index()
static_features_df

,product_id,cat1,cat2,cat3,brand,sku_size
0,20001,HC,ROPA LAVADO,Liquido,ARIEL,3000
1,20002,HC,ROPA LAVADO,Liquido,LIMPIEX,3000
2,20003,FOODS,ADEREZOS,Mayonesa,NATURA,475
3,20004,FOODS,ADEREZOS,Mayonesa,NATURA,240
4,20005,FOODS,ADEREZOS,Mayonesa,NATURA,120
...,...,...,...,...,...,...
775,21263,PC,CABELLO,SHAMPOO,VICHY,250
776,21265,PC,PIEL1,CUIDADO ESPECIAL,LANCOME,32
777,21266,PC,PIEL1,CUIDADO ESPECIAL,LANCOME,32
778,21267,PC,PIEL1,Cara,NIVEA,250


In [24]:
# uso el generador de features de autogluon para features sobre timestamp
#from autogluon.features.generators import DatetimeFeatureGenerator
#feature_generator = DatetimeFeatureGenerator()
#new_features = feature_generator.fit_transform(
#    df_monthly_product, target='cust_request_qty', timestamp='timestamp')
## mergeo las nuevas features con df_monthly_product usando el indice
#df_monthly_product = df_monthly_product.merge(
#    new_features, left_index=True, right_index=True)
#df_monthly_product

In [25]:
# rename timestamp_x with timestamp
#df_monthly_product = df_monthly_product.rename(
#    columns={'timestamp_x': 'timestamp', 'timestamp_y': 'timestamp_int'})

In [26]:
# ⏰ 4. Crear TimeSeriesDataFrame
ts_data = TimeSeriesDataFrame.from_data_frame(
    df_monthly_product,
    id_column='product_id',
    timestamp_column='timestamp',
    static_features_df=static_features_df,

)
ts_data = ts_data.fill_missing_values()

Trying to fill missing values in an unsorted dataframe. It is highly recommended to call `ts_df.sort_index()` before calling `ts_df.fill_missing_values()`


In [28]:
# ⚙️ 5. Definir y entrenar predictor
predictor = TimeSeriesPredictor(
    prediction_length=2,
    target='tn',
    freq='MS',  # Frecuencia mensual (Month Start)
    #known_covariates_names=["timestamp_int", "timestamp.year", "timestamp.month", "timestamp.day", "timestamp.dayofweek"]
)

predictor.fit(ts_data, num_val_windows=2, time_limit=60*60, refit_full=True, enable_ensemble=False, hyperparameters={"TemporalFusionTransformer": {}})

Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250703_232701'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       5.35 GB / 15.32 GB (34.9%)
Disk Space Avail:   60.31 GB / 575.67 GB (10.5%)

Fitting with arguments:
{'enable_ensemble': False,
 'eval_metric': WQL,
 'freq': 'MS',
 'hyperparameters': {'TemporalFusionTransformer': {}},
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': True,
 'skip_model_selection': False,
 'target': 'tn',
 'time_limit': 3600,
 'verbosity': 2}

train_data with frequency 

In [34]:
# 🔮 6. Generar predicción
forecast = predictor.predict(ts_data, model="TemporalFusionTransformer")

data with frequency 'IRREG' has been resampled to frequency 'MS'.


In [35]:
# Extraer predicción media y filtrar febrero 2020
forecast_mean = forecast['mean'].reset_index()
print(forecast_mean.columns)

Index(['item_id', 'timestamp', 'mean'], dtype='object')


In [36]:
# Tomar solo item_id y la predicción 'mean'
resultado = forecast['mean'].reset_index()[['item_id', 'mean']]
resultado.columns = ['product_id', 'tn']

# Filtrar solo febrero 2020
resultado = forecast['mean'].reset_index()
resultado = resultado[resultado['timestamp'] == '2020-02-01']

# Renombrar columnas
resultado = resultado[['item_id', 'mean']]
resultado.columns = ['product_id', 'tn']

In [37]:
# 💾 7. Guardar archivo
resultado.to_csv("predicciones_febrero2020_fecha_v02_02-base-features-tft.csv", index=False)
resultado.head()

,product_id,tn
1,20001,1349.482666
3,20002,995.452209
5,20003,787.043396
7,20004,591.864685
9,20005,538.085327
